# 메이플 월드 연구 — 본 수집

최근 랭킹에서 페이지를 무작위로 뽑고, 그 페이지의 캐릭터 200명 전원의 **생성일**을 받아 월드별로 저장합니다.

- **이어서 수집:** 진행 상황을 드라이브에 저장해서, 런타임이 끊기거나 다음 날 다시 실행해도 끝난 페이지는 건너뜁니다. 위에서부터 다시 실행하기만 하면 돼요.
- **키 종류:** 1번 셀의 `KEY_TYPE`만 바꾸면 호출 속도와 하루 한도가 맞춰집니다.
- **익명 저장:** 넥슨 안내(30일 넘게 보관할 자료는 개별 캐릭터를 알아볼 수 없게 익명 처리)에 맞춰, 수집하는 순간부터 익명 형태로만 저장합니다. 캐릭터명·식별자·랭킹 순위·페이지 번호는 남기지 않고, 레벨은 10단위 구간으로만 남깁니다. 페이지는 무작위 묶음 번호(cluster)로만 표시해, 표준오차 계산에 쓸 묶음 정보는 유지합니다.

- **생성 당시 월드(v4):** 캐릭터는 버닝 월드 리프·월드 리프로 월드를 옮길 수 있어서, 현재 월드가 생성한 월드와 다를 수 있습니다. 과거 날짜 기록은 띄엄띄엄 남아 있으므로, `LOOKUP_START`~`LOOKUP_END` 사이에 생성된 캐릭터는 생성 후 1·2·3·5·7·10·14·21·30·45·60일 뒤를 차례로 조회해 **처음 기록이 나온 날의 월드**(`world_first_seen`)와 **그 날짜**(`date_first_seen`)를 남깁니다. 2024-02-22(버닝 월드 리프 시작) 이전에 처음 잡힌 월드는 생성 월드로 확정할 수 있습니다.
- **저장 위치:** 이전 버전과 섞이지 않게 `maple_main_v4` 폴더에 저장합니다. 표집 계획(랭킹 기준일, 페이지 목록)은 v2의 `state.json`을 그대로 가져와 같은 표본을 씁니다.

**시작 전:** Colab 보안 비밀에 `NEXON_API_KEY`(받은 키)를 넣어 주세요. 키를 바꿨다면 값을 새 키로 바꿔 주세요.

## 1. 설정

In [ ]:
KEY_TYPE = 'dev'        # 개발 키: 'dev' / 서비스 키: 'service'

# 처치 월드(2024-01-12 해제)와 비교 월드
WORLDS = ['스카니아', '루나', '엘리시움', '크로아', '베라']

PAGES_PER_WORLD = 400   # 월드당 뽑을 랭킹 페이지 수(페이지당 200명). 400이면 월드당 약 8만 명
WINDOW_START, WINDOW_END = '2023-12-21', '2024-02-29'   # 분석 기간(요약용)
SEED = 2026

# 생성 당시 월드를 추가 조회할 생성일 범위 (API 과거 조회는 2023-12-21부터 가능)
LOOKUP_START, LOOKUP_END = '2023-12-21', '2024-03-31'
LOOKUP_OFFSETS = [1, 2, 3, 5, 7, 10, 14, 21, 30, 45, 60]   # 생성 후 며칠 뒤를 조회할지
LEAF_START = '2024-02-22'   # 버닝 월드 리프 시작일

LIMITS = {'dev': {'per_sec': 4, 'per_run': 950, 'workers': 1},
          'service': {'per_sec': 200, 'per_run': 15_000_000, 'workers': 16}}
PER_SEC, PER_RUN, WORKERS = (LIMITS[KEY_TYPE][k] for k in ('per_sec', 'per_run', 'workers'))
print(KEY_TYPE, '| 초당', PER_SEC, '| 이번 실행 상한', f'{PER_RUN:,}', '| 동시 요청', WORKERS)

## 2. 준비 (드라이브, 키, 호출 함수)

In [ ]:
import requests, time, random, json, os, csv, uuid, threading
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
from google.colab import userdata, drive

drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/maple_main_v4'
V2_STATE = '/content/drive/MyDrive/maple_main/state.json'
os.makedirs(OUT, exist_ok=True)
STATE = f'{OUT}/state.json'

API_KEY = (userdata.get('NEXON_API_KEY') or '').strip()
print('키 앞부분:', API_KEY[:5], '| 길이:', len(API_KEY))
BASE = 'https://open.api.nexon.com/maplestory/v1'
HEADERS = {'x-nxopen-api-key': API_KEY}

def band(level):
    try: l = int(level)
    except (TypeError, ValueError): return ''
    lo = l // 10 * 10
    return f'{lo}-{lo+9}'

class Budget(Exception): pass

_lock = threading.Lock()
_next_slot = [time.monotonic()]
calls = [0]
def _wait_turn():
    with _lock:
        if calls[0] >= PER_RUN:
            raise Budget()
        calls[0] += 1
        now = time.monotonic()
        slot = max(now, _next_slot[0])
        _next_slot[0] = slot + 1 / PER_SEC
    time.sleep(max(0, slot - now))

def get(path, **params):
    """API 1회 호출. 실패 시 {'_error': 코드}."""
    last429 = False
    for attempt in range(5):
        _wait_turn()
        try:
            r = requests.get(f'{BASE}{path}', headers=HEADERS, params=params, timeout=20)
            d = r.json()
        except Exception as e:
            time.sleep(2 ** attempt); continue
        if r.status_code == 429 or (r.status_code >= 500):
            last429 = r.status_code == 429
            time.sleep(2 ** attempt); continue
        if 'error' in d:
            return {'_error': d['error'].get('name', 'unknown')}
        return d
    if last429:   # 넥슨 쪽 하루 한도 초과 → 이 페이지는 완료 처리하지 않고 멈춤
        raise Budget()
    return {'_error': 'retry_exhausted'}

print('준비 완료. 키 길이:', len(API_KEY or ''))

## 3. 표집 계획 (처음 한 번만 실제로 계산)

랭킹 기준일을 정하고, 월드별 마지막 페이지를 찾은 뒤 무작위 페이지 목록을 만듭니다. 결과는 `state.json`에 저장되어, 다시 실행해도 같은 기준일과 같은 페이지를 씁니다.

In [ ]:
def load_state():
    return json.load(open(STATE)) if os.path.exists(STATE) else {}
def save_state(s):
    tmp = STATE + '.tmp'; json.dump(s, open(tmp, 'w'), ensure_ascii=False); os.replace(tmp, STATE)

state = load_state()
if 'rank_date' not in state and os.path.exists(V2_STATE):
    v2 = json.load(open(V2_STATE))   # v2와 같은 기준일·같은 페이지 표본을 쓰되, 진행 상황은 처음부터
    state = {'rank_date': v2['rank_date'],
             'worlds': {w: {'last_page': v['last_page'], 'pages': v['pages'], 'done': []} for w, v in v2['worlds'].items()}}
    save_state(state)
    print('v2 표집 계획을 가져왔습니다.')
if 'rank_date' not in state:
    state['rank_date'] = (date.today() - timedelta(days=1)).isoformat()
    state['worlds'] = {}
    save_state(state)
RANK_DATE = state['rank_date']
print('랭킹 기준일:', RANK_DATE)

def has_rows(world, page):
    d = get('/ranking/overall', date=RANK_DATE, world_name=world, page=page)
    if '_error' in d:
        raise RuntimeError(f'{world} {page}페이지 오류: {d["_error"]}')
    return bool(d.get('ranking'))

def find_last_page(world, cap=2**22):
    if not has_rows(world, 1): return 0
    lo, hi = 1, 2
    while hi <= cap and has_rows(world, hi):
        lo, hi = hi, hi * 2
    while hi - lo > 1:
        mid = (lo + hi) // 2
        if has_rows(world, mid): lo = mid
        else: hi = mid
    return lo

for w in WORLDS:
    ws = state['worlds'].get(w)
    if ws and ws.get('pages'):
        continue
    lp = find_last_page(w)
    rng = random.Random(f'{SEED}-{w}')
    pages = sorted(rng.sample(range(1, lp + 1), min(PAGES_PER_WORLD, lp)))
    state['worlds'][w] = {'last_page': lp, 'pages': pages, 'done': []}
    save_state(state)
    print(f'{w}: 마지막 페이지 {lp:,} (약 {lp*200:,}명) → {len(pages)}페이지 표집 | 누적 호출 {calls[0]}')

pd.DataFrame([{'world': w, 'last_page': v['last_page'], 'sampled_pages': len(v['pages']),
               'done_pages': len(v['done'])} for w, v in state['worlds'].items()])

## 4. 수집 (이어서 실행 가능)

페이지 하나 = 랭킹 1회 + 캐릭터 200명 × (식별자 1회 + 기본 정보 1회). 페이지가 끝날 때마다 익명 형태의 결과를 `records_<월드>.csv`에 붙이고 진행 상황을 저장합니다. 하루 한도에 닿으면 멈추니, 다음 날 같은 순서로 다시 실행하세요.

In [ ]:
# 저장 항목: 개별 캐릭터를 알아볼 수 있는 값(이름, 식별자, 순위, 페이지 번호, 정확한 레벨)은 남기지 않음
FIELDS = ['world', 'cluster', 'status', 'same_world', 'date_create', 'world_first_seen', 'date_first_seen', 'level_band_rank', 'level_band_now']
_YDAY = (date.today() - timedelta(days=1)).isoformat()

def first_seen(ocid, created):
    # 생성 후 LOOKUP_OFFSETS 순서로 조회해 처음 기록이 나온 (월드, 날짜). 끝내 없으면 ('', '')
    d0 = date.fromisoformat(created)
    for k in LOOKUP_OFFSETS:
        q = (d0 + timedelta(days=k)).isoformat()
        if q > _YDAY: break
        b = get('/character/basic', ocid=ocid, date=q)
        if '_error' not in b and b.get('world_name'):
            return b['world_name'], q
    return '', ''

def one_char(world, cluster, r):
    rec = {'world': world, 'cluster': cluster, 'status': '', 'same_world': '', 'date_create': '',
           'world_first_seen': '', 'date_first_seen': '', 'level_band_rank': band(r.get('character_level')), 'level_band_now': ''}
    idr = get('/id', character_name=r.get('character_name'))
    if '_error' in idr:
        rec['status'] = 'id_fail:' + idr['_error']; return rec
    b = get('/character/basic', ocid=idr['ocid'])
    if '_error' in b:
        rec['status'] = 'basic_fail:' + b['_error']; return rec
    rec.update(status='ok', same_world=int(b.get('world_name') == world),
               date_create=(b.get('character_date_create') or '')[:10],
               level_band_now=band(b.get('character_level')))
    if LOOKUP_START <= rec['date_create'] <= LOOKUP_END:
        rec['world_first_seen'], rec['date_first_seen'] = first_seen(idr['ocid'], rec['date_create'])
    return rec

def do_page(world, page, pool):
    d = get('/ranking/overall', date=RANK_DATE, world_name=world, page=page)
    if '_error' in d:
        raise RuntimeError(f'{world} {page}페이지 랭킹 오류: {d["_error"]}')
    rows = d.get('ranking', [])
    cluster = uuid.uuid4().hex[:10]   # 페이지 번호와 연결되지 않는 무작위 묶음 번호(어디에도 매핑 저장 안 함)
    recs = list(pool.map(lambda r: one_char(world, cluster, r), rows))
    random.shuffle(recs)              # 파일 안의 순서로 순위를 추정하지 못하게 섞음
    return recs

stopped = False
t0 = time.time()
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    for w in WORLDS:
        ws = state['worlds'][w]
        done = set(ws['done'])
        path = f'{OUT}/records_{w}.csv'
        for p in ws['pages']:
            if p in done: continue
            try:
                recs = do_page(w, p, pool)
            except Budget:
                stopped = True; break
            new = not os.path.exists(path)
            with open(path, 'a', newline='', encoding='utf-8-sig' if new else 'utf-8') as f:
                wr = csv.DictWriter(f, fieldnames=FIELDS)
                if new: wr.writeheader()
                wr.writerows(recs)
            ws['done'].append(p); done.add(p)
            save_state(state)
            if len(done) % 10 == 0:
                print(f'{w}: {len(done)}/{len(ws["pages"])}페이지 | 누적 호출 {calls[0]:,} | {time.time()-t0:,.0f}초')
        if stopped: break
        print(f'{w} 완료')

msg = '이번 실행 한도에 닿아 멈췄습니다. 다음에 다시 실행하면 이어서 수집합니다.' if stopped else '모든 월드 수집 완료.'
print(msg, '| 이번 실행 호출', f'{calls[0]:,}')

## 5. 진행 상황과 중간 요약

In [ ]:
frames = [pd.read_csv(f'{OUT}/records_{w}.csv') for w in WORLDS if os.path.exists(f'{OUT}/records_{w}.csv')]
if not frames:
    print('아직 수집된 기록이 없습니다.')
else:
    df = pd.concat(frames, ignore_index=True)
    df['date_create'] = pd.to_datetime(df['date_create'], errors='coerce')
    ok = df[df['status'] == 'ok']
    win = ok[(ok['date_create'] >= WINDOW_START) & (ok['date_create'] <= WINDOW_END)].copy()
    wf = win['world_first_seen'].fillna('').astype(str); df_ = win['date_first_seen'].fillna('').astype(str)
    win['origin'] = 'undetermined'
    win.loc[wf.str.startswith('버닝'), 'origin'] = 'burning'
    win.loc[(wf != '') & ~wf.str.startswith('버닝') & (df_ < LEAF_START), 'origin'] = 'confirmed'
    print('분석 기간 생성 캐릭터의 생성 월드 판별 결과 (confirmed=확정, burning=버닝 생성, undetermined=판별 불가)')
    print(win['origin'].value_counts())
    summary = pd.DataFrame({
        '수집 캐릭터': df.groupby('world').size(),
        '조회 성공': ok.groupby('world').size(),
        '분석 기간 생성': win.groupby('world').size(),
    }).fillna(0).astype(int)
    summary['완료 페이지'] = [len(state['worlds'][w]['done']) for w in summary.index]
    summary['표집 페이지'] = [len(state['worlds'][w]['pages']) for w in summary.index]
    print(summary)
    conf = win[win['origin'] == 'confirmed']
    daily = conf.groupby(['world_first_seen', conf['date_create'].dt.date]).size().unstack(0, fill_value=0)   # 생성 월드가 확정된 캐릭터만
    daily.to_csv(f'{OUT}/daily_creations_window.csv', encoding='utf-8-sig')
    print('\n분석 기간 일별 생성 건수 저장:', f'{OUT}/daily_creations_window.csv')